# Section 1: Project Setup
Set up libraries, file paths, and output folders for the federal contract spending cleaning pipeline.

In [39]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import gc

In [40]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis")

RAW_2021_DIR = BASE_DIR / "data_raw" / "FY2021_All_Contracts_Full_20260306"

files_2021 = sorted([p for p in RAW_2021_DIR.iterdir() if p.is_file()])

print(f"Files found: {len(files_2021)}")
for f in files_2021:
    print(f.name)

Files found: 7
FY2021_All_Contracts_Full_20260307_1.csv
FY2021_All_Contracts_Full_20260307_2.csv
FY2021_All_Contracts_Full_20260307_3.csv
FY2021_All_Contracts_Full_20260307_4.csv
FY2021_All_Contracts_Full_20260307_5.csv
FY2021_All_Contracts_Full_20260307_6.csv
FY2021_All_Contracts_Full_20260307_7.csv


# Section 2: Column Selection and Cleaning Rules
Define the subset of columns to retain and the rules for cleaning text, numeric, and date fields.

In [41]:
KEEP_COLS = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "transaction_number",
    "federal_action_obligation",
    "total_dollars_obligated",
    "base_and_exercised_options_value",
    "current_total_value_of_award",
    "action_date",
    "action_date_fiscal_year",
    "period_of_performance_start_date",
    "period_of_performance_current_end_date",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_parent_name",
    "recipient_state_code",
    "recipient_state_name",
    "primary_place_of_performance_state_code",
    "primary_place_of_performance_state_name",
    "award_type",
    "award_type_code",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "type_of_set_aside",
    "number_of_offers_received",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink",
    "last_modified_date"
]

In [42]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(" ", "_", regex=False)
    )
    return df


def clean_text_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            )
    return df


def convert_numeric_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def convert_date_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

In [43]:
TEXT_COLS = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "transaction_number",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_parent_name",
    "recipient_state_code",
    "recipient_state_name",
    "primary_place_of_performance_state_code",
    "primary_place_of_performance_state_name",
    "award_type",
    "award_type_code",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "type_of_set_aside",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink"
]

NUMERIC_COLS = [
    "federal_action_obligation",
    "total_dollars_obligated",
    "base_and_exercised_options_value",
    "current_total_value_of_award",
    "number_of_offers_received",
    "action_date_fiscal_year"
]

DATE_COLS = [
    "action_date",
    "period_of_performance_start_date",
    "period_of_performance_current_end_date",
    "last_modified_date"
]

In [44]:
test_file = files_2021[0]
print("Testing file:", test_file.name)

df_test = pd.read_csv(
    test_file,
    usecols=lambda c: c in KEEP_COLS,
    low_memory=False
)

df_test = clean_column_names(df_test)

print(df_test.shape)
print(df_test.columns.tolist())
df_test.head()

Testing file: FY2021_All_Contracts_Full_20260307_1.csv
(1000000, 38)
['contract_transaction_unique_key', 'contract_award_unique_key', 'award_id_piid', 'modification_number', 'transaction_number', 'federal_action_obligation', 'total_dollars_obligated', 'base_and_exercised_options_value', 'current_total_value_of_award', 'action_date', 'action_date_fiscal_year', 'period_of_performance_start_date', 'period_of_performance_current_end_date', 'awarding_agency_name', 'awarding_sub_agency_name', 'awarding_office_name', 'funding_agency_name', 'funding_sub_agency_name', 'recipient_name', 'recipient_parent_name', 'recipient_state_code', 'recipient_state_name', 'primary_place_of_performance_state_code', 'primary_place_of_performance_state_name', 'award_type_code', 'award_type', 'type_of_contract_pricing', 'transaction_description', 'product_or_service_code', 'product_or_service_code_description', 'naics_code', 'naics_description', 'extent_competed', 'type_of_set_aside', 'number_of_offers_received',

,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,transaction_number,federal_action_obligation,total_dollars_obligated,base_and_exercised_options_value,current_total_value_of_award,action_date,...,product_or_service_code,product_or_service_code_description,naics_code,naics_description,extent_competed,type_of_set_aside,number_of_offers_received,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,0.0,-6277.55,5490842.19,-6277.55,5490842.19,2021-09-30,...,S206,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,NaN,NaN,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.0,0.00,310000.00,0.00,310000.00,2021-09-30,...,R608,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,0.0,34260.25,34260.25,34260.25,34260.25,2021-09-30,...,7E20,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,NaN,7.0,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,0.0,248.00,248.00,248.00,248.00,2021-09-30,...,5120,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,0.0,876.00,876.00,876.00,876.00,2021-09-30,...,5120,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00


# Section 3: Cleaning Test on a Sample File
Load a sample contract file, apply initial cleaning steps, and validate column structure, data types, and missing values before scaling to full-file processing.

In [45]:
# Clean text columns
text_cols = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "transaction_number",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_parent_name",
    "recipient_state_code",
    "recipient_state_name",
    "primary_place_of_performance_state_code",
    "primary_place_of_performance_state_name",
    "award_type",
    "award_type_code",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "type_of_set_aside",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink"
]

numeric_cols = [
    "federal_action_obligation",
    "total_dollars_obligated",
    "base_and_exercised_options_value",
    "current_total_value_of_award",
    "number_of_offers_received",
    "action_date_fiscal_year"
]

date_cols = [
    "action_date",
    "period_of_performance_start_date",
    "period_of_performance_current_end_date",
    "last_modified_date"
]

for col in text_cols:
    if col in df_test.columns:
        df_test[col] = df_test[col].astype("string").str.strip()

for col in numeric_cols:
    if col in df_test.columns:
        df_test[col] = pd.to_numeric(df_test[col], errors="coerce")

for col in date_cols:
    if col in df_test.columns:
        df_test[col] = pd.to_datetime(df_test[col], errors="coerce")

# Drop duplicate transactions
df_test = df_test.drop_duplicates(subset=["contract_transaction_unique_key"])

# Identify and drop columns with no data at all
all_null_cols = df_test.columns[df_test.isna().all()].tolist()
df_test = df_test.drop(columns=all_null_cols)

print(df_test.shape)
print("Dropped all-null columns:", all_null_cols)
df_test.head()

(1000000, 38)
Dropped all-null columns: []


,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,transaction_number,federal_action_obligation,total_dollars_obligated,base_and_exercised_options_value,current_total_value_of_award,action_date,...,product_or_service_code,product_or_service_code_description,naics_code,naics_description,extent_competed,type_of_set_aside,number_of_offers_received,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,0.0,-6277.55,5490842.19,-6277.55,5490842.19,2021-09-30,...,S206,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,<NA>,NaN,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00:00
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.0,0.00,310000.00,0.00,310000.00,2021-09-30,...,R608,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,<NA>,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00:00
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,0.0,34260.25,34260.25,34260.25,34260.25,2021-09-30,...,7E20,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,<NA>,7.0,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00:00
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,0.0,248.00,248.00,248.00,248.00,2021-09-30,...,5120,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,<NA>,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00:00
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,0.0,876.00,876.00,876.00,876.00,2021-09-30,...,5120,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,<NA>,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00:00


In [46]:
print("Data types:")
display(df_test.dtypes)

print("\nMissing values:")
display(df_test.isna().sum().sort_values(ascending=False))

print("\nFiscal year counts:")
display(df_test["action_date_fiscal_year"].value_counts(dropna=False).sort_index())

print("\nTop agencies by federal_action_obligation:")
top_agencies = (
    df_test.groupby("awarding_agency_name", dropna=False)["federal_action_obligation"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
display(top_agencies)

Data types:


contract_transaction_unique_key                             string[python]
contract_award_unique_key                                   string[python]
award_id_piid                                               string[python]
modification_number                                         string[python]
transaction_number                                          string[python]
federal_action_obligation                                          float64
total_dollars_obligated                                            float64
base_and_exercised_options_value                                   float64
current_total_value_of_award                                       float64
action_date                                                 datetime64[ns]
action_date_fiscal_year                                              int64
period_of_performance_start_date                            datetime64[ns]
period_of_performance_current_end_date                      datetime64[ns]
awarding_agency_name     


Missing values:


type_of_set_aside                                      774665
number_of_offers_received                              674318
primary_place_of_performance_state_name                 66434
primary_place_of_performance_state_code                 66434
current_total_value_of_award                            36168
award_type                                              36168
award_type_code                                         36168
period_of_performance_current_end_date                  36168
base_and_exercised_options_value                        36168
transaction_number                                      36168
recipient_state_code                                    20894
recipient_state_name                                     8043
extent_competed                                          2842
recipient_parent_name                                     343
product_or_service_code_description                       223
transaction_description                                    88
type_of_


Fiscal year counts:


action_date_fiscal_year
2021    1000000
Name: count, dtype: int64


Top agencies by federal_action_obligation:


,awarding_agency_name,federal_action_obligation
0,Department of Defense,7.083536e+10
1,Department of Veterans Affairs,1.670596e+10
2,Department of Health and Human Services,9.704225e+09
3,Department of Homeland Security,6.066585e+09
4,General Services Administration,4.319596e+09
5,Department of State,4.094588e+09
6,National Aeronautics and Space Administration,3.275290e+09
7,Department of Energy,2.822930e+09
8,Department of Agriculture,2.015458e+09
9,Department of the Interior,1.973140e+09


In [47]:
missing_summary = pd.DataFrame({
    "missing_count": df_test.isna().sum(),
    "missing_pct": (df_test.isna().sum() / len(df_test)) * 100
}).sort_values("missing_pct", ascending=False)

missing_summary

,missing_count,missing_pct
type_of_set_aside,774665,77.4665
number_of_offers_received,674318,67.4318
primary_place_of_performance_state_name,66434,6.6434
primary_place_of_performance_state_code,66434,6.6434
current_total_value_of_award,36168,3.6168
award_type,36168,3.6168
award_type_code,36168,3.6168
period_of_performance_current_end_date,36168,3.6168
base_and_exercised_options_value,36168,3.6168
transaction_number,36168,3.6168


# Section 4: Full Cleaning Pipeline for One 2021 Raw File
Run chunk-based cleaning on one complete raw 2021 contract file and save the cleaned output to the processed data folder.

In [48]:
## Define chunk processing function

def process_large_csv_in_chunks(file_path, output_path, chunk_size=200_000, fiscal_year=2021):
    first_write = True
    total_rows_written = 0

    for chunk in pd.read_csv(
        file_path,
        usecols=lambda c: c in KEEP_COLS,
        chunksize=chunk_size,
        low_memory=False
    ):
        # Clean text columns
        for col in TEXT_COLS:
            if col in chunk.columns:
                chunk[col] = chunk[col].astype("string").str.strip()

        # Clean numeric columns
        for col in NUMERIC_COLS:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        # Clean date columns
        for col in DATE_COLS:
            if col in chunk.columns:
                chunk[col] = pd.to_datetime(chunk[col], errors="coerce")

        # Add source fiscal year column
        chunk["source_fiscal_year"] = fiscal_year

        # Drop duplicate transactions within chunk
        if "contract_transaction_unique_key" in chunk.columns:
            chunk = chunk.drop_duplicates(subset=["contract_transaction_unique_key"])
        else:
            chunk = chunk.drop_duplicates()

        # Drop columns that are completely empty within the chunk
        # (This is safe, but most/all useful columns should remain)
        chunk = chunk.dropna(axis=1, how="all")

        total_rows_written += len(chunk)

        # Write chunk to output file
        if first_write:
            chunk.to_csv(output_path, index=False, mode="w")
            first_write = False
        else:
            chunk.to_csv(output_path, index=False, mode="a", header=False)

        del chunk
        gc.collect()

    print(f"Finished processing: {file_path.name}")
    print(f"Saved cleaned file to: {output_path}")
    print(f"Total rows written: {total_rows_written}")

In [49]:
print(f"Number of raw 2021 files found: {len(files_2021)}")
for i, f in enumerate(files_2021, start=1):
    print(i, f.name)

Number of raw 2021 files found: 7
1 FY2021_All_Contracts_Full_20260307_1.csv
2 FY2021_All_Contracts_Full_20260307_2.csv
3 FY2021_All_Contracts_Full_20260307_3.csv
4 FY2021_All_Contracts_Full_20260307_4.csv
5 FY2021_All_Contracts_Full_20260307_5.csv
6 FY2021_All_Contracts_Full_20260307_6.csv
7 FY2021_All_Contracts_Full_20260307_7.csv


In [50]:
## Select one full 2021 file
full_file_2021 = files_2021[0]
print("Selected file:", full_file_2021.name)

Selected file: FY2021_All_Contracts_Full_20260307_1.csv


In [51]:
## Define output path and run full-file cleaning
output_file_2021 = PROCESSED_DIR / f"{full_file_2021.stem}_clean.csv"
print("Output path:", output_file_2021)

Output path: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2021_All_Contracts_Full_20260307_1_clean.csv


In [52]:
process_large_csv_in_chunks(
    file_path=full_file_2021,
    output_path=output_file_2021,
    chunk_size=200_000,
    fiscal_year=2021
)

Finished processing: FY2021_All_Contracts_Full_20260307_1.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2021_All_Contracts_Full_20260307_1_clean.csv
Total rows written: 1000000


In [53]:
## Load the cleaned file for validation
df_2021_file1_clean = pd.read_csv(output_file_2021, low_memory=False)

print("Cleaned file shape:", df_2021_file1_clean.shape)
df_2021_file1_clean.head()

Cleaned file shape: (1000000, 39)


,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,transaction_number,federal_action_obligation,total_dollars_obligated,base_and_exercised_options_value,current_total_value_of_award,action_date,...,product_or_service_code_description,naics_code,naics_description,extent_competed,type_of_set_aside,number_of_offers_received,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date,source_fiscal_year
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,0.0,-6277.55,5490842.19,-6277.55,5490842.19,2021-09-30,...,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,NaN,NaN,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00:00,2021
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.0,0.00,310000.00,0.00,310000.00,2021-09-30,...,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00:00,2021
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,0.0,34260.25,34260.25,34260.25,34260.25,2021-09-30,...,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,NaN,7.0,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00:00,2021
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,0.0,248.00,248.00,248.00,248.00,2021-09-30,...,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00:00,2021
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,0.0,876.00,876.00,876.00,876.00,2021-09-30,...,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00:00,2021


In [54]:
## Validate the cleaned output
print("Columns:")
print(df_2021_file1_clean.columns.tolist())

Columns:
['contract_transaction_unique_key', 'contract_award_unique_key', 'award_id_piid', 'modification_number', 'transaction_number', 'federal_action_obligation', 'total_dollars_obligated', 'base_and_exercised_options_value', 'current_total_value_of_award', 'action_date', 'action_date_fiscal_year', 'period_of_performance_start_date', 'period_of_performance_current_end_date', 'awarding_agency_name', 'awarding_sub_agency_name', 'awarding_office_name', 'funding_agency_name', 'funding_sub_agency_name', 'recipient_name', 'recipient_parent_name', 'recipient_state_code', 'recipient_state_name', 'primary_place_of_performance_state_code', 'primary_place_of_performance_state_name', 'award_type_code', 'award_type', 'type_of_contract_pricing', 'transaction_description', 'product_or_service_code', 'product_or_service_code_description', 'naics_code', 'naics_description', 'extent_competed', 'type_of_set_aside', 'number_of_offers_received', 'contracting_officers_determination_of_business_size', 'usa

In [55]:
print("Source fiscal year counts:")
print(df_2021_file1_clean["source_fiscal_year"].value_counts(dropna=False))

Source fiscal year counts:
source_fiscal_year
2021    1000000
Name: count, dtype: int64


In [56]:
missing_summary_file1 = pd.DataFrame({
    "missing_count": df_2021_file1_clean.isna().sum(),
    "missing_pct": (df_2021_file1_clean.isna().sum() / len(df_2021_file1_clean)) * 100
}).sort_values("missing_pct", ascending=False)

missing_summary_file1

,missing_count,missing_pct
type_of_set_aside,774665,77.4665
number_of_offers_received,674318,67.4318
primary_place_of_performance_state_name,66434,6.6434
primary_place_of_performance_state_code,66434,6.6434
period_of_performance_current_end_date,36168,3.6168
current_total_value_of_award,36168,3.6168
award_type,36168,3.6168
award_type_code,36168,3.6168
base_and_exercised_options_value,36168,3.6168
transaction_number,36168,3.6168


In [57]:
top_agencies_file1 = (
    df_2021_file1_clean.groupby("awarding_agency_name")["federal_action_obligation"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_agencies_file1

,awarding_agency_name,federal_action_obligation
0,Department of Defense,7.083536e+10
1,Department of Veterans Affairs,1.670596e+10
2,Department of Health and Human Services,9.704225e+09
3,Department of Homeland Security,6.066585e+09
4,General Services Administration,4.319596e+09
5,Department of State,4.094588e+09
6,National Aeronautics and Space Administration,3.275290e+09
7,Department of Energy,2.822930e+09
8,Department of Agriculture,2.015458e+09
9,Department of the Interior,1.973140e+09


In [58]:
## save a lean version of the cleaned file
lean_cols = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "federal_action_obligation",
    "total_dollars_obligated",
    "action_date",
    "action_date_fiscal_year",
    "source_fiscal_year",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_state_name",
    "primary_place_of_performance_state_name",
    "award_type",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink",
    "last_modified_date"
]

df_2021_file1_lean = df_2021_file1_clean[
    [col for col in lean_cols if col in df_2021_file1_clean.columns]
].copy()

lean_output_file_2021 = PROCESSED_DIR / f"{full_file_2021.stem}_clean_lean.csv"
df_2021_file1_lean.to_csv(lean_output_file_2021, index=False)

print("Saved lean cleaned file to:", lean_output_file_2021)
print("Lean shape:", df_2021_file1_lean.shape)

Saved lean cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2021_All_Contracts_Full_20260307_1_clean_lean.csv
Lean shape: (1000000, 28)


# Section 5: Full Cleaning for All 2021 Files
Apply the chunk-based cleaning pipeline to all raw 2021 contract files and save cleaned outputs for each file.

In [59]:
## Process all 2021 raw files

processed_files_2021 = []

for file_path in files_2021:
    print(f"Processing raw file: {file_path.name}")
    
    output_file = PROCESSED_DIR / f"{file_path.stem}_clean.csv"
    
    process_large_csv_in_chunks(
        file_path=file_path,
        output_path=output_file,
        chunk_size=200_000,
        fiscal_year=2021
    )
    
    processed_files_2021.append(output_file)

print("\nFinished processing all 2021 raw files.")
print("Processed files created:")
for f in processed_files_2021:
    print(f.name)

Processing raw file: FY2021_All_Contracts_Full_20260307_1.csv
Finished processing: FY2021_All_Contracts_Full_20260307_1.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2021_All_Contracts_Full_20260307_1_clean.csv
Total rows written: 1000000
Processing raw file: FY2021_All_Contracts_Full_20260307_2.csv
Finished processing: FY2021_All_Contracts_Full_20260307_2.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2021_All_Contracts_Full_20260307_2_clean.csv
Total rows written: 1000000
Processing raw file: FY2021_All_Contracts_Full_20260307_3.csv
Finished processing: FY2021_All_Contracts_Full_20260307_3.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\d

In [60]:
## Create lean versions for all cleaned 2021 files
lean_cols = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "federal_action_obligation",
    "total_dollars_obligated",
    "action_date",
    "action_date_fiscal_year",
    "source_fiscal_year",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_state_name",
    "primary_place_of_performance_state_name",
    "award_type",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink",
    "last_modified_date"
]

lean_files_2021 = []

for clean_file in processed_files_2021:
    print(f"Creating lean file for: {clean_file.name}")
    
    df_clean = pd.read_csv(clean_file, low_memory=False)
    
    df_lean = df_clean[[col for col in lean_cols if col in df_clean.columns]].copy()
    
    lean_output = PROCESSED_DIR / f"{clean_file.stem}_lean.csv"
    df_lean.to_csv(lean_output, index=False)
    
    lean_files_2021.append(lean_output)
    
    del df_clean, df_lean
    gc.collect()

print("\nFinished creating lean files for all 2021 cleaned outputs.")
for f in lean_files_2021:
    print(f.name)

Creating lean file for: FY2021_All_Contracts_Full_20260307_1_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_2_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_3_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_4_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_5_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_6_clean.csv
Creating lean file for: FY2021_All_Contracts_Full_20260307_7_clean.csv

Finished creating lean files for all 2021 cleaned outputs.
FY2021_All_Contracts_Full_20260307_1_clean_lean.csv
FY2021_All_Contracts_Full_20260307_2_clean_lean.csv
FY2021_All_Contracts_Full_20260307_3_clean_lean.csv
FY2021_All_Contracts_Full_20260307_4_clean_lean.csv
FY2021_All_Contracts_Full_20260307_5_clean_lean.csv
FY2021_All_Contracts_Full_20260307_6_clean_lean.csv
FY2021_All_Contracts_Full_20260307_7_clean_lean.csv


In [61]:
## Quick validation of all 2021 cleaned files
for clean_file in processed_files_2021:
    df_check = pd.read_csv(clean_file, low_memory=False)
    
    print(f"\nFile: {clean_file.name}")
    print("Shape:", df_check.shape)
    
    if "source_fiscal_year" in df_check.columns:
        print("Source fiscal year counts:")
        print(df_check["source_fiscal_year"].value_counts(dropna=False))
    
    if {"awarding_agency_name", "federal_action_obligation"}.issubset(df_check.columns):
        print("Top 5 agencies by federal_action_obligation:")
        print(
            df_check.groupby("awarding_agency_name")["federal_action_obligation"]
            .sum()
            .sort_values(ascending=False)
            .head(5)
        )
    
    del df_check
    gc.collect()


File: FY2021_All_Contracts_Full_20260307_1_clean.csv
Shape: (1000000, 39)
Source fiscal year counts:
source_fiscal_year
2021    1000000
Name: count, dtype: int64
Top 5 agencies by federal_action_obligation:
awarding_agency_name
Department of Defense                      7.083536e+10
Department of Veterans Affairs             1.670596e+10
Department of Health and Human Services    9.704225e+09
Department of Homeland Security            6.066585e+09
General Services Administration            4.319596e+09
Name: federal_action_obligation, dtype: float64

File: FY2021_All_Contracts_Full_20260307_2_clean.csv
Shape: (1000000, 39)
Source fiscal year counts:
source_fiscal_year
2021    1000000
Name: count, dtype: int64
Top 5 agencies by federal_action_obligation:
awarding_agency_name
Department of Defense                            5.739857e+10
Department of Veterans Affairs                   7.367680e+09
Department of Health and Human Services          7.132292e+09
Department of Energy        

# Section 6: Combine All Cleaned 2021 Files
Merge the cleaned 2021 contract files into one consolidated year-level dataset for downstream analysis.

In [62]:
## Combine full cleaned 2021 files
clean_files_2021 = sorted(PROCESSED_DIR.glob("FY2021*_clean.csv"))

print("Cleaned 2021 files found:")
for f in clean_files_2021:
    print(f.name)

Cleaned 2021 files found:
FY2021_All_Contracts_Full_20260307_1_clean.csv
FY2021_All_Contracts_Full_20260307_2_clean.csv
FY2021_All_Contracts_Full_20260307_3_clean.csv
FY2021_All_Contracts_Full_20260307_4_clean.csv
FY2021_All_Contracts_Full_20260307_5_clean.csv
FY2021_All_Contracts_Full_20260307_6_clean.csv
FY2021_All_Contracts_Full_20260307_7_clean.csv


In [63]:
df_2021_all = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in clean_files_2021],
    ignore_index=True
)

print("Combined 2021 full cleaned shape:", df_2021_all.shape)
df_2021_all.head()

Combined 2021 full cleaned shape: (6405238, 39)


,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,transaction_number,federal_action_obligation,total_dollars_obligated,base_and_exercised_options_value,current_total_value_of_award,action_date,...,product_or_service_code_description,naics_code,naics_description,extent_competed,type_of_set_aside,number_of_offers_received,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date,source_fiscal_year
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,0.0,-6277.55,5490842.19,-6277.55,5490842.19,2021-09-30,...,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,NaN,NaN,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00:00,2021
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.0,0.00,310000.00,0.00,310000.00,2021-09-30,...,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00:00,2021
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,0.0,34260.25,34260.25,34260.25,34260.25,2021-09-30,...,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,NaN,7.0,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00:00,2021
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,0.0,248.00,248.00,248.00,248.00,2021-09-30,...,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00:00,2021
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,0.0,876.00,876.00,876.00,876.00,2021-09-30,...,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,NaN,NaN,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00:00,2021


In [64]:
output_2021_all = PROCESSED_DIR / "contracts_2021_all_clean.csv"
df_2021_all.to_csv(output_2021_all, index=False)

print("Saved combined 2021 full cleaned file to:", output_2021_all)

Saved combined 2021 full cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2021_all_clean.csv


In [65]:
## Combine lean cleaned 2021 files
lean_files_2021 = sorted(PROCESSED_DIR.glob("FY2021*_clean_lean.csv"))

print("Lean 2021 files found:")
for f in lean_files_2021:
    print(f.name)

Lean 2021 files found:
FY2021_All_Contracts_Full_20260307_1_clean_lean.csv
FY2021_All_Contracts_Full_20260307_2_clean_lean.csv
FY2021_All_Contracts_Full_20260307_3_clean_lean.csv
FY2021_All_Contracts_Full_20260307_4_clean_lean.csv
FY2021_All_Contracts_Full_20260307_5_clean_lean.csv
FY2021_All_Contracts_Full_20260307_6_clean_lean.csv
FY2021_All_Contracts_Full_20260307_7_clean_lean.csv


In [66]:
df_2021_all_lean = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in lean_files_2021],
    ignore_index=True
)

print("Combined 2021 lean shape:", df_2021_all_lean.shape)
df_2021_all_lean.head()

Combined 2021 lean shape: (6405238, 28)


,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,federal_action_obligation,total_dollars_obligated,action_date,action_date_fiscal_year,source_fiscal_year,awarding_agency_name,...,type_of_contract_pricing,transaction_description,product_or_service_code,product_or_service_code_description,naics_code,naics_description,extent_competed,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,-6277.55,5490842.19,2021-09-30,2021,2021,Government Accountability Office,...,FIRM FIXED PRICE,THIS $0 RQG IS TO DEOBLIGATE PRIOR YEAR FUNDS ...,S206,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00:00
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.00,310000.00,2021-09-30,2021,2021,Executive Office of the President,...,LABOR HOURS,INTERPRETING SERVICES FOR THE WHITE HOUSE,R608,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00:00
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,34260.25,34260.25,2021-09-30,2021,2021,Department of Agriculture,...,FIRM FIXED PRICE,PROCURE MONITORS FOR FNS REMOTE WORK. F1 21 ...,7E20,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00:00
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,248.00,248.00,2021-09-30,2021,2021,General Services Administration,...,FIRM FIXED PRICE,MAIL CART,5120,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00:00
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,876.00,876.00,2021-09-30,2021,2021,General Services Administration,...,FIRM FIXED PRICE,INDUSTRIAL HAND SOAP,5120,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00:00


In [67]:
output_2021_all_lean = PROCESSED_DIR / "contracts_2021_all_clean_lean.csv"
df_2021_all_lean.to_csv(output_2021_all_lean, index=False)

print("Saved combined 2021 lean cleaned file to:", output_2021_all_lean)

Saved combined 2021 lean cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2021_all_clean_lean.csv


# Section 7: Repeat Cleaning Pipeline for 2022–2024
Apply the same cleaning and consolidation workflow to the 2022, 2023, and 2024 contract data files.

In [68]:
## Define raw folders for 2022–2024
RAW_2022_DIR = BASE_DIR / "data_raw" / "FY2022_All_Contracts_Full_20260306"
RAW_2023_DIR = BASE_DIR / "data_raw" / "FY2023_All_Contracts_Full_20260306"
RAW_2024_DIR = BASE_DIR / "data_raw" / "FY2024_All_Contracts_Full_20260306"

year_folder_map = {
    2022: RAW_2022_DIR,
    2023: RAW_2023_DIR,
    2024: RAW_2024_DIR
}

for year, folder in year_folder_map.items():
    print(year, folder, "| exists =", folder.exists())

2022 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data_raw\FY2022_All_Contracts_Full_20260306 | exists = True
2023 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data_raw\FY2023_All_Contracts_Full_20260306 | exists = True
2024 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data_raw\FY2024_All_Contracts_Full_20260306 | exists = True


In [69]:
## Build file lists for each year
files_by_year = {}

for year, folder in year_folder_map.items():
    files = sorted([p for p in folder.iterdir() if p.is_file()])
    files_by_year[year] = files
    
    print(f"\nYear: {year}")
    print(f"Files found: {len(files)}")
    for f in files:
        print(" -", f.name)


Year: 2022
Files found: 7
 - FY2022_All_Contracts_Full_20260307_1.csv
 - FY2022_All_Contracts_Full_20260307_2.csv
 - FY2022_All_Contracts_Full_20260307_3.csv
 - FY2022_All_Contracts_Full_20260307_4.csv
 - FY2022_All_Contracts_Full_20260307_5.csv
 - FY2022_All_Contracts_Full_20260307_6.csv
 - FY2022_All_Contracts_Full_20260307_7.csv

Year: 2023
Files found: 7
 - FY2023_All_Contracts_Full_20260307_1.csv
 - FY2023_All_Contracts_Full_20260307_2.csv
 - FY2023_All_Contracts_Full_20260307_3.csv
 - FY2023_All_Contracts_Full_20260307_4.csv
 - FY2023_All_Contracts_Full_20260307_5.csv
 - FY2023_All_Contracts_Full_20260307_6.csv
 - FY2023_All_Contracts_Full_20260307_7.csv

Year: 2024
Files found: 7
 - FY2024_All_Contracts_Full_20260307_1.csv
 - FY2024_All_Contracts_Full_20260307_2.csv
 - FY2024_All_Contracts_Full_20260307_3.csv
 - FY2024_All_Contracts_Full_20260307_4.csv
 - FY2024_All_Contracts_Full_20260307_5.csv
 - FY2024_All_Contracts_Full_20260307_6.csv
 - FY2024_All_Contracts_Full_20260307_7

In [70]:
## Process all raw files for 2022–2024
processed_files_by_year = {}

for year, files in files_by_year.items():
    print(f"\nProcessing all raw files for {year}...")
    
    processed_files = []
    
    for file_path in files:
        print(f"Processing raw file: {file_path.name}")
        
        output_file = PROCESSED_DIR / f"{file_path.stem}_clean.csv"
        
        process_large_csv_in_chunks(
            file_path=file_path,
            output_path=output_file,
            chunk_size=200_000,
            fiscal_year=year
        )
        
        processed_files.append(output_file)
    
    processed_files_by_year[year] = processed_files

print("\nFinished processing all raw files for 2022–2024.")


Processing all raw files for 2022...
Processing raw file: FY2022_All_Contracts_Full_20260307_1.csv
Finished processing: FY2022_All_Contracts_Full_20260307_1.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2022_All_Contracts_Full_20260307_1_clean.csv
Total rows written: 1000000
Processing raw file: FY2022_All_Contracts_Full_20260307_2.csv
Finished processing: FY2022_All_Contracts_Full_20260307_2.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\FY2022_All_Contracts_Full_20260307_2_clean.csv
Total rows written: 1000000
Processing raw file: FY2022_All_Contracts_Full_20260307_3.csv
Finished processing: FY2022_All_Contracts_Full_20260307_3.csv
Saved cleaned file to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\

In [72]:
## Create lean cleaned files for 2022–2024
lean_cols = [
    "contract_transaction_unique_key",
    "contract_award_unique_key",
    "award_id_piid",
    "modification_number",
    "federal_action_obligation",
    "total_dollars_obligated",
    "action_date",
    "action_date_fiscal_year",
    "source_fiscal_year",
    "awarding_agency_name",
    "awarding_sub_agency_name",
    "awarding_office_name",
    "funding_agency_name",
    "funding_sub_agency_name",
    "recipient_name",
    "recipient_state_name",
    "primary_place_of_performance_state_name",
    "award_type",
    "type_of_contract_pricing",
    "transaction_description",
    "product_or_service_code",
    "product_or_service_code_description",
    "naics_code",
    "naics_description",
    "extent_competed",
    "contracting_officers_determination_of_business_size",
    "usaspending_permalink",
    "last_modified_date"
]

lean_files_by_year = {}

for year, processed_files in processed_files_by_year.items():
    print(f"\nCreating lean cleaned files for {year}...")
    
    lean_files = []
    
    for clean_file in processed_files:
        print(f"Creating lean file for: {clean_file.name}")
        
        df_clean = pd.read_csv(clean_file, low_memory=False)
        df_lean = df_clean[[col for col in lean_cols if col in df_clean.columns]].copy()
        
        lean_output = PROCESSED_DIR / f"{clean_file.stem}_lean.csv"
        df_lean.to_csv(lean_output, index=False)
        
        lean_files.append(lean_output)
        
        del df_clean, df_lean
        gc.collect()
    
    lean_files_by_year[year] = lean_files

print("\nFinished creating lean cleaned files for 2022–2024.")


Creating lean cleaned files for 2022...
Creating lean file for: FY2022_All_Contracts_Full_20260307_1_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_2_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_3_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_4_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_5_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_6_clean.csv
Creating lean file for: FY2022_All_Contracts_Full_20260307_7_clean.csv

Creating lean cleaned files for 2023...
Creating lean file for: FY2023_All_Contracts_Full_20260307_1_clean.csv
Creating lean file for: FY2023_All_Contracts_Full_20260307_2_clean.csv
Creating lean file for: FY2023_All_Contracts_Full_20260307_3_clean.csv
Creating lean file for: FY2023_All_Contracts_Full_20260307_4_clean.csv
Creating lean file for: FY2023_All_Contracts_Full_20260307_5_clean.csv
Creating lean file for: FY2023_All_Contracts_Full_20260307_6_clean

In [74]:
## Combine cleaned files into one big file for each year
yearly_full_outputs = {}
yearly_lean_outputs = {}

for year in [2022, 2023, 2024]:
    print(f"\nCombining cleaned files for {year}...")
    
    full_files = processed_files_by_year[year]
    lean_files = lean_files_by_year[year]
    
    df_year_full = pd.concat(
        [pd.read_csv(f, low_memory=False) for f in full_files],
        ignore_index=True
    )
    
    df_year_lean = pd.concat(
        [pd.read_csv(f, low_memory=False) for f in lean_files],
        ignore_index=True
    )
    
    full_output = PROCESSED_DIR / f"contracts_{year}_all_clean.csv"
    lean_output = PROCESSED_DIR / f"contracts_{year}_all_clean_lean.csv"
    
    df_year_full.to_csv(full_output, index=False)
    df_year_lean.to_csv(lean_output, index=False)
    
    yearly_full_outputs[year] = full_output
    yearly_lean_outputs[year] = lean_output
    
    print(f"Saved full cleaned yearly file: {full_output}")
    print(f"Saved lean cleaned yearly file: {lean_output}")
    print(f"{year} full shape: {df_year_full.shape}")
    print(f"{year} lean shape: {df_year_lean.shape}")
    
    del df_year_full, df_year_lean
    gc.collect()


Combining cleaned files for 2022...
Saved full cleaned yearly file: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2022_all_clean.csv
Saved lean cleaned yearly file: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2022_all_clean_lean.csv
2022 full shape: (6667358, 39)
2022 lean shape: (6667358, 28)

Combining cleaned files for 2023...
Saved full cleaned yearly file: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2023_all_clean.csv
Saved lean cleaned yearly file: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2023_all_clean_lean.csv
2023 full shape: (6693427, 39)
2023 lean shape

In [75]:
## Validate yearly outputs
for year in [2022, 2023, 2024]:
    print(f"\nValidating yearly output for {year}...")
    
    df_year = pd.read_csv(yearly_full_outputs[year], low_memory=False)
    
    print("Shape:", df_year.shape)
    print("Source fiscal year counts:")
    print(df_year["source_fiscal_year"].value_counts(dropna=False))
    
    print("\nTop 5 agencies by federal_action_obligation:")
    print(
        df_year.groupby("awarding_agency_name")["federal_action_obligation"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )
    
    del df_year
    gc.collect()


Validating yearly output for 2022...
Shape: (6667358, 39)
Source fiscal year counts:
source_fiscal_year
2022    6667358
Name: count, dtype: int64

Top 5 agencies by federal_action_obligation:
awarding_agency_name
Department of Defense                      4.143901e+11
Department of Veterans Affairs             5.638916e+10
Department of Energy                       4.289786e+10
Department of Health and Human Services    3.888827e+10
General Services Administration            2.118697e+10
Name: federal_action_obligation, dtype: float64

Validating yearly output for 2023...
Shape: (6693427, 39)
Source fiscal year counts:
source_fiscal_year
2023    6693427
Name: count, dtype: int64

Top 5 agencies by federal_action_obligation:
awarding_agency_name
Department of Defense                      4.568266e+11
Department of Veterans Affairs             6.170215e+10
Department of Energy                       4.627435e+10
Department of Health and Human Services    3.850180e+10
General Services Adm

# Section 8: Build Master Cleaned Dataset (2021–2024)
Combine the cleaned yearly datasets into one master federal contract spending dataset for analysis and dashboard development.

In [76]:
## Define yearly cleaned files
yearly_full_files = {
    2021: PROCESSED_DIR / "contracts_2021_all_clean.csv",
    2022: PROCESSED_DIR / "contracts_2022_all_clean.csv",
    2023: PROCESSED_DIR / "contracts_2023_all_clean.csv",
    2024: PROCESSED_DIR / "contracts_2024_all_clean.csv"
}

yearly_lean_files = {
    2021: PROCESSED_DIR / "contracts_2021_all_clean_lean.csv",
    2022: PROCESSED_DIR / "contracts_2022_all_clean_lean.csv",
    2023: PROCESSED_DIR / "contracts_2023_all_clean_lean.csv",
    2024: PROCESSED_DIR / "contracts_2024_all_clean_lean.csv"
}

for year, file_path in yearly_full_files.items():
    print(year, file_path, "| exists =", file_path.exists())

2021 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2021_all_clean.csv | exists = True
2022 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2022_all_clean.csv | exists = True
2023 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2023_all_clean.csv | exists = True
2024 C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2024_all_clean.csv | exists = True


In [78]:
## Combine all cleaned yearly files in chunks
master_full_output = PROCESSED_DIR / "contracts_2021_2024_all_clean.csv"

# start fresh
if master_full_output.exists():
    master_full_output.unlink()

first_write = True

for year, file_path in yearly_full_files.items():
    print(f"Appending year {year}: {file_path.name}")

    for chunk in pd.read_csv(file_path, chunksize=200_000, low_memory=False):
        if first_write:
            chunk.to_csv(master_full_output, index=False, mode="w")
            first_write = False
        else:
            chunk.to_csv(master_full_output, index=False, mode="a", header=False)

        del chunk
        gc.collect()

print("Saved master full cleaned dataset to:", master_full_output)

Appending year 2021: contracts_2021_all_clean.csv
Appending year 2022: contracts_2022_all_clean.csv
Appending year 2023: contracts_2023_all_clean.csv
Appending year 2024: contracts_2024_all_clean.csv
Saved master full cleaned dataset to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2021_2024_all_clean.csv


In [79]:
## Combine all lean cleaned yearly files
master_lean_output = PROCESSED_DIR / "contracts_2021_2024_all_clean_lean.csv"

# start fresh
if master_lean_output.exists():
    master_lean_output.unlink()

first_write = True

for year, file_path in yearly_lean_files.items():
    print(f"Appending lean year {year}: {file_path.name}")

    for chunk in pd.read_csv(file_path, chunksize=200_000, low_memory=False):
        if first_write:
            chunk.to_csv(master_lean_output, index=False, mode="w")
            first_write = False
        else:
            chunk.to_csv(master_lean_output, index=False, mode="a", header=False)

        del chunk
        gc.collect()

print("Saved master lean cleaned dataset to:", master_lean_output)

Appending lean year 2021: contracts_2021_all_clean_lean.csv
Appending lean year 2022: contracts_2022_all_clean_lean.csv
Appending lean year 2023: contracts_2023_all_clean_lean.csv
Appending lean year 2024: contracts_2024_all_clean_lean.csv
Saved master lean cleaned dataset to: C:\Users\AKKem\OneDrive\Desktop\Data Analysis Modules\Projects\federal_spending_analysis\federal-spending-efficiency-analysis\data\processed\contracts_2021_2024_all_clean_lean.csv


# Section 9: Final Data Quality Checks
Validate the combined 2021–2024 master datasets using chunk-based checks for row counts, fiscal year counts, missing values, and top agency spending totals.

In [81]:
## Validate row count and fiscal year distribution for master full file
row_count_full = 0
year_counts_full = {}

for chunk in pd.read_csv(master_full_output, chunksize=200_000, low_memory=False):
    row_count_full += len(chunk)

    counts = chunk["source_fiscal_year"].value_counts(dropna=False).to_dict()
    for year, count in counts.items():
        year_counts_full[year] = year_counts_full.get(year, 0) + count

    del chunk
    gc.collect()

print("Master full total rows:", row_count_full)
print("Master full source fiscal year counts:")
print(dict(sorted(year_counts_full.items())))

Master full total rows: 26456744
Master full source fiscal year counts:
{2021: 6405238, 2022: 6667358, 2023: 6693427, 2024: 6690721}


In [82]:
## Validate row count and fiscal year distribution for master lean file
row_count_lean = 0
year_counts_lean = {}

for chunk in pd.read_csv(master_lean_output, chunksize=200_000, low_memory=False):
    row_count_lean += len(chunk)

    counts = chunk["source_fiscal_year"].value_counts(dropna=False).to_dict()
    for year, count in counts.items():
        year_counts_lean[year] = year_counts_lean.get(year, 0) + count

    del chunk
    gc.collect()

print("Master lean total rows:", row_count_lean)
print("Master lean source fiscal year counts:")
print(dict(sorted(year_counts_lean.items())))

Master lean total rows: 26456744
Master lean source fiscal year counts:
{2021: 6405238, 2022: 6667358, 2023: 6693427, 2024: 6690721}


In [83]:
## Top 10 agencies by federal_action_obligation from master full file
agency_totals_full = {}

for chunk in pd.read_csv(master_full_output, chunksize=200_000, low_memory=False):
    grouped = chunk.groupby("awarding_agency_name")["federal_action_obligation"].sum()

    for agency, total in grouped.items():
        agency_totals_full[agency] = agency_totals_full.get(agency, 0) + total

    del chunk
    gc.collect()

top_agencies_full = pd.Series(agency_totals_full).sort_values(ascending=False).head(10)

print("Top 10 agencies by federal_action_obligation:")
print(top_agencies_full)

Top 10 agencies by federal_action_obligation:
Department of Defense                            1.704101e+12
Department of Veterans Affairs                   2.326825e+11
Department of Energy                             1.754033e+11
Department of Health and Human Services          1.529807e+11
General Services Administration                  9.088732e+10
Department of Homeland Security                  8.869040e+10
National Aeronautics and Space Administration    8.009558e+10
Department of State                              4.551684e+10
Department of Agriculture                        4.309357e+10
Department of Justice                            3.880626e+10
dtype: float64


In [84]:
## Total obligations by fiscal year from master full file
year_obligation_totals = {}

for chunk in pd.read_csv(master_full_output, chunksize=200_000, low_memory=False):
    grouped = chunk.groupby("source_fiscal_year")["federal_action_obligation"].sum()

    for year, total in grouped.items():
        year_obligation_totals[year] = year_obligation_totals.get(year, 0) + total

    del chunk
    gc.collect()

year_obligation_totals = dict(sorted(year_obligation_totals.items()))

print("Total federal_action_obligation by fiscal year:")
print(year_obligation_totals)

Total federal_action_obligation by fiscal year:
{2021: 645626628857.7999, 2022: 694625533522.8901, 2023: 760358735579.7299, 2024: 756204897071.2101}


In [85]:
## Missingness check on master lean file
from collections import defaultdict

missing_counts = defaultdict(int)
total_rows = 0
all_columns = None

for chunk in pd.read_csv(master_lean_output, chunksize=200_000, low_memory=False):
    total_rows += len(chunk)

    if all_columns is None:
        all_columns = chunk.columns.tolist()

    nulls = chunk.isna().sum().to_dict()
    for col, count in nulls.items():
        missing_counts[col] += count

    del chunk
    gc.collect()

missing_summary_master_lean = pd.DataFrame({
    "missing_count": [missing_counts[col] for col in all_columns],
    "missing_pct": [(missing_counts[col] / total_rows) * 100 for col in all_columns]
}, index=all_columns).sort_values("missing_pct", ascending=False)

missing_summary_master_lean

,missing_count,missing_pct
primary_place_of_performance_state_name,1750562,6.616695
award_type,986474,3.728630
recipient_state_name,249846,0.944357
extent_competed,85658,0.323766
product_or_service_code_description,1905,0.007200
transaction_description,1730,0.006539
type_of_contract_pricing,1470,0.005556
naics_description,1181,0.004464
naics_code,1179,0.004456
product_or_service_code,255,0.000964


In [86]:
## preview first few rows of master lean file
master_lean_preview = pd.read_csv(master_lean_output, nrows=5, low_memory=False)
master_lean_preview

,contract_transaction_unique_key,contract_award_unique_key,award_id_piid,modification_number,federal_action_obligation,total_dollars_obligated,action_date,action_date_fiscal_year,source_fiscal_year,awarding_agency_name,...,type_of_contract_pricing,transaction_description,product_or_service_code,product_or_service_code_description,naics_code,naics_description,extent_competed,contracting_officers_determination_of_business_size,usaspending_permalink,last_modified_date
0,0559_0559_GAO15SB00060001_P00007_GAO15SB0006_0,CONT_AWD_GAO15SB00060001_0559_GAO15SB0006_0559,GAO15SB00060001,P00007,-6277.55,5490842.19,2021-09-30,2021,2021,Government Accountability Office,...,FIRM FIXED PRICE,THIS $0 RQG IS TO DEOBLIGATE PRIOR YEAR FUNDS ...,S206,HOUSEKEEPING- GUARD,561612.0,SECURITY GUARDS AND PATROL SERVICES,FULL AND OPEN COMPETITION,OTHER THAN SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_GAO...,2021-09-30 09:46:43+00:00
1,1100_1100_11316021F0003WHO_P00003_11316021D100...,CONT_AWD_11316021F0003WHO_1100_11316021D1004WH...,11316021F0003WHO,P00003,0.00,310000.00,2021-09-30,2021,2021,Executive Office of the President,...,LABOR HOURS,INTERPRETING SERVICES FOR THE WHITE HOUSE,R608,SUPPORT- ADMINISTRATIVE: TRANSLATION AND INTER...,541930.0,TRANSLATION AND INTERPRETATION SERVICES,NOT AVAILABLE FOR COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_113...,2021-09-30 15:45:41+00:00
2,1205_8000_12314421F0895_0_NNG15SC79B_0,CONT_AWD_12314421F0895_1205_NNG15SC79B_8000,12314421F0895,0,34260.25,34260.25,2021-09-30,2021,2021,Department of Agriculture,...,FIRM FIXED PRICE,PROCURE MONITORS FOR FNS REMOTE WORK. F1 21 ...,7E20,"IT AND TELECOM - END USER: HELP DESK;TIER 1-2,...",541519.0,OTHER COMPUTER RELATED SERVICES,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_123...,2025-04-16 16:16:15+00:00
3,4732_4732_47QSWA21F84VB_0_GS07Q1647AJN2E_0,CONT_AWD_47QSWA21F84VB_4732_GS07Q1647AJN2E_4732,47QSWA21F84VB,0,248.00,248.00,2021-09-30,2021,2021,General Services Administration,...,FIRM FIXED PRICE,MAIL CART,5120,"HAND TOOLS, NONEDGED, NONPOWERED",444130.0,HARDWARE STORES,FULL AND OPEN COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-03-05 11:16:49+00:00
4,4732_4732_47QSWA21F84ZA_0_47QSWA20A0012_0,CONT_AWD_47QSWA21F84ZA_4732_47QSWA20A0012_4732,47QSWA21F84ZA,0,876.00,876.00,2021-09-30,2021,2021,General Services Administration,...,FIRM FIXED PRICE,INDUSTRIAL HAND SOAP,5120,"HAND TOOLS, NONEDGED, NONPOWERED",332510.0,HARDWARE MANUFACTURING,FULL AND OPEN COMPETITION,SMALL BUSINESS,https://www.usaspending.gov/award/CONT_AWD_47Q...,2023-10-30 10:13:34+00:00
